### Build Constructors Dimension

In [0]:
%run ../00-common/01-environment-config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
from pyspark.sql import functions as F

### 1. Read source tables
silver.constructors, gold.ref_natioanlity_region

In [0]:
constructors_df = spark.table(f"{catalog_name}.{silver_schema}.constructors")
nationality_region_ref_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
display(constructors_df)
display(nationality_region_ref_df)

constructor_id,name,nationality,ingestion_timestamp,source_file
behra-porsche,Behra-porsche,Italian,2026-07-28T10:52:43.797Z,dbfs:/Volumes/formula1/landing/files/constructors.json
cisitalia,Cisitalia,Italian,2026-07-28T10:52:43.797Z,dbfs:/Volumes/formula1/landing/files/constructors.json
elder,Elder,American,2026-07-28T10:52:43.797Z,dbfs:/Volumes/formula1/landing/files/constructors.json
haas,Haas F1 Team,American,2026-07-28T10:52:43.797Z,dbfs:/Volumes/formula1/landing/files/constructors.json
hall,Hall,American,2026-07-28T10:52:43.797Z,dbfs:/Volumes/formula1/landing/files/constructors.json
klenk,Klenk,German,2026-07-28T10:52:43.797Z,dbfs:/Volumes/formula1/landing/files/constructors.json
lago,Talbot-lago,French,2026-07-28T10:52:43.797Z,dbfs:/Volumes/formula1/landing/files/constructors.json
lesovsky,Lesovsky,American,2026-07-28T10:52:43.797Z,dbfs:/Volumes/formula1/landing/files/constructors.json
life,Life,Italian,2026-07-28T10:52:43.797Z,dbfs:/Volumes/formula1/landing/files/constructors.json
lotus-borgward,Lotus-borgward,British,2026-07-28T10:52:43.797Z,dbfs:/Volumes/formula1/landing/files/constructors.json


nationality,region
British,Europe
Italian,Europe
French,Europe
German,Europe
Swiss,Europe
Dutch,Europe
Belgium,Europe
Belgian,Europe
Irish,Europe
Spanish,Europe


### 2. Join two tables

In [0]:
dim_constructor_df = (
    constructors_df
    .join(nationality_region_ref_df,
            on = "nationality",
            how = "left")
    .select(
        constructors_df.constructor_id,
        constructors_df.name,
        constructors_df.nationality,
        nationality_region_ref_df.region.alias("nationality_region")
    )
)
display(dim_constructor_df)

constructor_id,name,nationality,nationality_region
behra-porsche,Behra-porsche,Italian,Europe
cisitalia,Cisitalia,Italian,Europe
elder,Elder,American,North America
haas,Haas F1 Team,American,North America
hall,Hall,American,North America
klenk,Klenk,German,Europe
lago,Talbot-lago,French,Europe
lesovsky,Lesovsky,American,North America
life,Life,Italian,Europe
lotus-borgward,Lotus-borgward,British,Europe


### Writing data to gold table

In [0]:
(
    dim_constructor_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(target_table)
    )
display(spark.table(target_table))

constructor_id,name,nationality,nationality_region
behra-porsche,Behra-porsche,Italian,Europe
cisitalia,Cisitalia,Italian,Europe
elder,Elder,American,North America
haas,Haas F1 Team,American,North America
hall,Hall,American,North America
klenk,Klenk,German,Europe
lago,Talbot-lago,French,Europe
lesovsky,Lesovsky,American,North America
life,Life,Italian,Europe
lotus-borgward,Lotus-borgward,British,Europe
